# 📊 Notebook 01 — Exploratory Data Analysis
**Financial Fraud Detection System**

This notebook performs a thorough EDA of the credit card transactions dataset,
examines the severe class imbalance, and surfaces key patterns in fraudulent behaviour.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
df = pd.read_csv("../data/raw_data.csv")
print(f"Shape: {df.shape}")
df.head()

## 1. Basic Data Quality

In [ ]:
print("=== Missing values ===")
print(df.isnull().sum().sum(), "total missing values")

print("\n=== Duplicate rows ===")
print(df.duplicated().sum())

print("\n=== Data types ===")
print(df.dtypes.value_counts())

print("\n=== Descriptive statistics (Amount) ===")
print(df["Amount"].describe())

## 2. Class Imbalance

In [ ]:
class_counts = df["Class"].value_counts()
fraud_pct = df["Class"].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count bar
bars = axes[0].bar(["Legitimate", "Fraud"], class_counts.values,
                   color=["#27AE60", "#C0392B"], edgecolor="white", width=0.5)
axes[0].bar_label(bars, labels=[f"{v:,}" for v in class_counts.values],
                  padding=5, fontsize=11, fontweight="bold")
axes[0].set_title("Transaction Count by Class", fontweight="bold")
axes[0].set_ylabel("Count")

# Pie
axes[1].pie(class_counts.values,
            labels=[f"Legitimate\n{class_counts[0]:,}", f"Fraud\n{class_counts[1]:,}"],
            autopct="%1.3f%%", colors=["#27AE60", "#C0392B"],
            startangle=90, pctdistance=0.85)
axes[1].set_title(f"Class Distribution\n(Fraud ≈ {fraud_pct:.4f}%)", fontweight="bold")

plt.tight_layout()
plt.savefig("../outputs/graphs/01_class_distribution.png", bbox_inches="tight")
plt.show()

print(f"\nFraud rate: {fraud_pct:.4f}%  →  This is a highly imbalanced dataset.")
print("Strategy: Apply SMOTE during model training to balance class distribution.")

## 3. Transaction Amount Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, class_val, label, colour in zip(
        axes, [0, 1], ["Legitimate", "Fraud"], ["#27AE60", "#C0392B"]):
    data = df[df["Class"] == class_val]["Amount"]
    ax.hist(data, bins=80, color=colour, edgecolor="white", alpha=0.85)
    ax.set_title(f"{label} Transactions — Amount Distribution", fontweight="bold")
    ax.set_xlabel("Amount (USD)")
    ax.set_ylabel("Frequency")
    stats = f"Mean: ${data.mean():.2f}\nMedian: ${data.median():.2f}\nMax: ${data.max():.2f}"
    ax.text(0.7, 0.85, stats, transform=ax.transAxes,
            bbox=dict(boxstyle="round", fc="white", alpha=0.8))

plt.suptitle("Amount Distribution: Legitimate vs Fraud", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("../outputs/graphs/01_amount_distribution.png", bbox_inches="tight")
plt.show()

print("\nKey insight: Fraudulent transactions tend to cluster at lower amounts,")
print("suggesting fraudsters prefer smaller transactions to avoid detection.")

## 4. Time-Based Fraud Trends

In [ ]:
df["hour_of_day"] = (df["Time"] // 3600 % 24).astype(int)

hourly = df.groupby("hour_of_day").agg(
    total=("Class", "count"),
    fraud=("Class", "sum")
).reset_index()
hourly["fraud_rate"] = hourly["fraud"] / hourly["total"] * 100

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Total volume
axes[0].fill_between(hourly["hour_of_day"], hourly["total"],
                     alpha=0.4, color="#2980B9")
axes[0].plot(hourly["hour_of_day"], hourly["total"],
             color="#2980B9", lw=2, marker="o", ms=4)
axes[0].set_title("Transaction Volume by Hour", fontweight="bold")
axes[0].set_ylabel("Count")

# Fraud rate
axes[1].fill_between(hourly["hour_of_day"], hourly["fraud_rate"],
                     alpha=0.4, color="#C0392B")
axes[1].plot(hourly["hour_of_day"], hourly["fraud_rate"],
             color="#C0392B", lw=2, marker="o", ms=4)
axes[1].axhspan(0, hourly["fraud_rate"].mean(), alpha=0.08, color="grey")
axes[1].set_title("Fraud Rate (%) by Hour", fontweight="bold")
axes[1].set_ylabel("Fraud Rate (%)")
axes[1].set_xlabel("Hour of Day (0 = Midnight)")
axes[1].set_xticks(range(24))

plt.tight_layout()
plt.savefig("../outputs/graphs/01_time_analysis.png", bbox_inches="tight")
plt.show()

peak_hour = hourly.loc[hourly["fraud_rate"].idxmax(), "hour_of_day"]
print(f"\nPeak fraud hour: {peak_hour}:00  ({hourly['fraud_rate'].max():.4f}% fraud rate)")

## 5. PCA Feature Correlation Heatmap

In [ ]:
v_cols = [f"V{i}" for i in range(1, 15)]   # first 14 PCA features for readability
corr = df[v_cols + ["Amount", "Class"]].corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap="RdBu_r", center=0,
            annot=True, fmt=".2f", linewidths=0.5,
            annot_kws={"size": 8}, ax=ax)
ax.set_title("Feature Correlation Heatmap (V1–V14 + Amount + Class)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/graphs/01_correlation_heatmap.png", bbox_inches="tight")
plt.show()

# Most correlated features with Class
top_corr = corr["Class"].abs().sort_values(ascending=False).drop("Class").head(10)
print("\nTop 10 features correlated with fraud:\n", top_corr)

## 6. Box Plot — Amount by Class

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

df.boxplot(column="Amount", by="Class", ax=axes[0],
           boxprops=dict(color="#2C3E50"),
           medianprops=dict(color="#E74C3C", linewidth=2))
axes[0].set_title("Amount Distribution by Class")
axes[0].set_xlabel("Class (0=Legit, 1=Fraud)")
axes[0].set_ylabel("Amount (USD)")

# Log-scale version for clarity
df["log_amount"] = np.log1p(df["Amount"])
df.boxplot(column="log_amount", by="Class", ax=axes[1],
           boxprops=dict(color="#2C3E50"),
           medianprops=dict(color="#E74C3C", linewidth=2))
axes[1].set_title("log(Amount+1) by Class")
axes[1].set_xlabel("Class (0=Legit, 1=Fraud)")

plt.suptitle("")
plt.tight_layout()
plt.savefig("../outputs/graphs/01_boxplot_amount.png", bbox_inches="tight")
plt.show()

## EDA Summary

| Finding | Detail |
|---|---|
| Severe class imbalance | Fraud = 0.17% — SMOTE required |
| Fraud amount pattern | Lower median amount vs. legitimate |
| Peak fraud hours | Late night (0–4 AM) shows highest fraud rate |
| Key PCA features | V17, V14, V12 most correlated with fraud |

**Next step:** `02_feature_engineering.ipynb` — create features and apply SMOTE